In [ ]:
#pip install streamlit pandas scikit-learn requests opencc-python-reimplemented

In [ ]:
import streamlit as st


In [ ]:
DEFAULT_CSV_PATH = "complaints_final.csv"

In [ ]:
# =========================
# UI
# =========================
st.title("🤖 PM小幫手")
st.caption("輸入問題後，系統會根據既有客訴案例與處理方式，整理出自然語言建議。")

csv_path = DEFAULT_CSV_PATH

question = st.text_area(
    "請輸入你的問題",
    placeholder="例如：消費者反映付款後沒有收到正確結果，該怎麼排查？",
    height=100
)

col1, col2 = st.columns([1, 5]) #會把畫面橫向切成多欄，左右欄寬比例是 1:5
with col1:
    run_btn = st.button("開始分析", use_container_width=True) #把按鈕放在左邊col1，按鈕寬度會撐滿它所在的容器(True)

if run_btn: #按下按鈕時
    if not question.strip(): #如果去掉空白後是空字串，就代表沒有有效輸入
        st.error("請先輸入問題。") #顯示紅色錯誤訊息
    else:
        status = st.empty() #空白顯示區塊，裝下一部後面的info

        try:
            status.info("正在讀取資料...")
            df = load_data(csv_path.strip()) #自定義的load_data函式

            status.info("正在檢索相似案例...")
            top_results = rank_cases(question, df, top_k=3)

            status.info("正在整理回答...")
            prompt = build_prompt(question, top_results)
            answer = ask_ollama(prompt)

            status.success("分析完成")

            st.subheader("小幫手回覆")
            st.write(answer)

            with st.expander("查看參考來源"): #可展開、收合區塊
                for i, row in top_results.iterrows():
                    solution = str(row["Solution in Chinese"]).replace("建議處理方案（專業版）：", "").strip()
                    st.markdown(f"### 來源 {i+1}")
                    st.write(solution)
                    st.caption(
                        f"Issue: {row['Issue']} | Sub-issue: {row['Sub-issue']} | "
                        f"score={row['final_score']:.4f}"
                    )

        except FileNotFoundError:
            status.empty()
            st.error("找不到 CSV 檔案，請檢查預設路徑是否正確。")
        except Exception as e:
            status.empty()
            st.error(f"執行時發生錯誤：{e}")